# 03. Baseline: SVM

## import tools.py và models.py

In [10]:
import sys, json
sys.path.insert(0, "..") 
import numpy as np, pandas as pd
from src.tools import load_features, tune_walk_forward, evaluate_on_test, summarize, train_test
from src.models import SVM

data = load_features()
print(data.shape)

(2478, 15)


## Lưới tham số và chọn tham số

In [11]:
GRID = [
    {"kernel": ["linear"], "C": [0.01, 0.1, 1, 10]},
    {"kernel": ["rbf"], "C": [0.1, 1, 10], "gamma": [0.01, 0.05, 0.2]},
]
best, rows = tune_walk_forward(SVM, GRID, data.loc[:"2022"])
json.dump({"best": best, "rows": rows}, open("../reports/tables/tuning_SVM.json", "w"), indent=1)
print("Tham số tốt nhất:", best)
pd.DataFrame([{**r["params"], "BalAcc_val": r["BalAcc"], "std": r["std"]} for r in rows]) \
  .sort_values("BalAcc_val", ascending=False).head(8)

Tham số tốt nhất: {'kernel': 'linear', 'C': 0.01}


,kernel,C,BalAcc_val,std,gamma
0,linear,0.01,0.517547,0.046325,NaN
7,rbf,1.00,0.517101,0.024260,0.01
4,rbf,0.10,0.514011,0.030548,0.01
5,rbf,0.10,0.512811,0.021273,0.05
6,rbf,0.10,0.503542,0.011309,0.20
12,rbf,10.00,0.502120,0.018560,0.20
3,linear,10.00,0.500396,0.023129,NaN
1,linear,0.10,0.499205,0.039018,NaN


## Đánh giá trên test

In [12]:
res = evaluate_on_test("SVM", SVM, best, data)
res.to_csv("../reports/tables/test_SVM.csv", index=False)
summarize(res, ["Accuracy", "BalAcc", "F1", "AUC", "MCC", "UpRate", "time_s", "nonzero"])

,Accuracy,BalAcc,F1,AUC,MCC,UpRate,time_s,nonzero
model,,,,,,,,
SVM,0.5154,0.5127,0.5585,0.5028,0.0250,0.5194,0.0332,0.9700


## Đọc nghiệm qua điều kiện KKT

In [13]:
X_train, y_train, X_test, y_test, *_ = train_test(data)
m = SVM(**best).fit(X_train, y_train)
alpha = np.abs(m.model_.dual_coef_.ravel())      # |alpha_i * y_i| = alpha_i của các vector hỗ trợ
C_i = best["C"] * m.cw_[m.model_.support_]       # cận trên của từng alpha_i
kkt = pd.Series({
    "alpha = 0 (ngoài lề)": len(y_train) - len(alpha),
    "0 < alpha < C (đúng trên lề)": int(np.sum(alpha < C_i - 1e-8)),
    "alpha = C (vi phạm lề)": int(np.sum(alpha >= C_i - 1e-8)),
})
kkt.to_csv("../reports/tables/svm_kkt.csv")
kkt

alpha = 0 (ngoài lề)              52
0 < alpha < C (đúng trên lề)       9
alpha = C (vi phạm lề)          1670
dtype: int64